In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
import importlib
import My_Pipeline
importlib.reload(My_Pipeline)
from My_Pipeline import my_pipeline

randomstate = 42

In [290]:
model_ready_df = my_pipeline('raw_data')
model_ready_df.head()

,station_Møllendalsplass,station_Torgallmenningen,station_Grieghallen,station_Høyteknologisenteret,station_Studentboligene,station_Akvariet,station_Damsgårdsveien 71,station_Dreggsallmenningen Sør,station_Florida Bybanestopp,timestamp,station,free_bikes,temperature,precipitation,wind_speed,flow,hour,is_weekend,free_bikes_plus_1h
0,0,0,0,0,0,0,0,1,0,2024-09-01 02:00:00+00:00,Dreggsallmenningen Sør,7,7.1,0.0,4.0,0,2,1,7
1,0,0,0,0,0,0,0,1,0,2024-09-01 03:00:00+00:00,Dreggsallmenningen Sør,7,5.8,0.0,3.0,4,3,1,6
2,0,0,0,0,0,0,0,1,0,2024-09-01 04:00:00+00:00,Dreggsallmenningen Sør,6,5.3,0.0,1.5,16,4,1,7
3,0,0,0,1,0,0,0,0,0,2024-09-01 04:00:00+00:00,Høyteknologisenteret,0,5.3,0.0,1.5,16,4,1,0
4,1,0,0,0,0,0,0,0,0,2024-09-01 05:00:00+00:00,Møllendalsplass,11,5.3,0.0,1.5,22,5,1,10


In [291]:
X = model_ready_df.iloc[:, :-1]
y = model_ready_df.iloc[:, -1]
X = X.drop(columns=['timestamp','station'])


In [292]:
X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, random_state=randomstate, train_size=.6, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_test_val, y_test_val, train_size=.5, shuffle=False)

In [293]:
y_test_pred = X_test["free_bikes"]
rmse_baseline = mean_squared_error(y_test, y_test_pred)**.5

print(f"Baseline RMSE (neste = nåværende): {rmse_baseline:.3f}")

Baseline RMSE (neste = nåværende): 1.824


In [294]:
models = {}
hgb_pipe = Pipeline([
    ("gb", HistGradientBoostingRegressor(
        max_iter=500,
        early_stopping=False,
        random_state=randomstate,
    )),
])

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "gb__learning_rate": [0.03, 0.05, 0.1],
    "gb__max_depth": [None, 6, 12],
    "gb__min_samples_leaf": [20, 50],
    "gb__l2_regularization": [0.0, 0.1],
}

gs_hgb = GridSearchCV(
    estimator=hgb_pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=0,
)

gs_hgb.fit(X_train, y_train)

best_hgb = gs_hgb.best_estimator_
cv_rmse_hgb = (-gs_hgb.best_score_) ** 0.5
models[best_hgb] = cv_rmse_hgb

print("Best params (HGB):", gs_hgb.best_params_)
print(f"CV RMSE (best):       {cv_rmse_hgb:.4f}")

Best params (HGB): {'gb__l2_regularization': 0.1, 'gb__learning_rate': 0.03, 'gb__max_depth': None, 'gb__min_samples_leaf': 50}
CV RMSE (best):       1.5878


In [295]:
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", Ridge(alpha=1.0, random_state=randomstate))
])

param_grid = {
    "reg__alpha": np.logspace(-2, 2, 9)
}

gs_ridge = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=0,
)

gs_ridge.fit(X_train, y_train)

best_ridge = gs_ridge.best_estimator_
cv_rmse_ridge = (-gs_ridge.best_score_) ** 0.5

models[best_ridge] = cv_rmse_ridge

print("Best params (Ridge):", gs_ridge.best_params_)
print(f"CV RMSE (best):       {cv_rmse_ridge:.4f}")

Best params (Ridge): {'reg__alpha': np.float64(0.01)}
CV RMSE (best):       1.7644


In [296]:
knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

param_grid = {
    "knn__n_neighbors": [3, 5, 10, 20],
    "knn__weights": ["uniform", "distance"],
}

gs_knn = GridSearchCV(
    estimator=knn_pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=0,
)

gs_knn.fit(X_train, y_train)

best_knn = gs_knn.best_estimator_
cv_rmse_knn = (-gs_knn.best_score_) ** 0.5
models[best_knn] = cv_rmse_knn

print("Best params (KNN):", gs_knn.best_params_)
print(f"CV RMSE (best):    {cv_rmse_knn:.4f}")

Best params (KNN): {'knn__n_neighbors': 10, 'knn__weights': 'distance'}
CV RMSE (best):    3.3152


In [297]:
best_model = min(models, key=models.get)

y_val_pred = best_model.predict(X_val)
best_model_val_rmse = mean_squared_error(y_val, y_val_pred)**.5
print(best_model)
print(best_model_val_rmse)

Pipeline(steps=[('gb',
                 HistGradientBoostingRegressor(early_stopping=False,
                                               l2_regularization=0.1,
                                               learning_rate=0.03, max_iter=500,
                                               min_samples_leaf=50,
                                               random_state=42))])
1.1137905783263158


In [305]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_val,
    y_val,
    n_repeats=20,
    scoring="neg_mean_squared_error",
    random_state=randomstate,
    n_jobs=-1,
)
delta_mse = perm.importances_mean   
delta_std = perm.importances_std

fi = pd.DataFrame({
    "feature": X_val.columns,
    "delta_MSE": perm.importances_mean,
    "std": perm.importances_std
}).sort_values("delta_MSE", ascending=False)

fi.head(16)

,feature,delta_MSE,std
9,free_bikes,14.916068,0.204777
14,hour,0.398113,0.034359
13,flow,0.174685,0.008577
3,station_Høyteknologisenteret,0.164255,0.010647
8,station_Florida Bybanestopp,0.048188,0.003447
5,station_Akvariet,0.020293,0.005132
0,station_Møllendalsplass,0.014376,0.004246
6,station_Damsgårdsveien 71,0.014298,0.002407
1,station_Torgallmenningen,0.012468,0.002722
4,station_Studentboligene,0.011378,0.001977


In [299]:
# Hent ut kolonner med høyest importance
top_features = fi['feature'].head(3).tolist()
print("Top 3 viktigste features:")
print(top_features)

Top 3 viktigste features:
['free_bikes', 'hour', 'flow']


In [300]:
X_train_top, X_val_top, X_test_top= X_train[top_features], X_val[top_features], X_test[top_features]

In [ ]:
from sklearn.base import clone

best_model_top = clone(best_model)
best_model_top.fit(X_train_top, y_train)
y_val_pred_top = best_model_top.predict(X_val_top)
bm_val_rmse_top = mean_squared_error(y_val, y_val_pred_top)**.5
print(best_model_top)
print(bm_val_rmse_top)

Pipeline(steps=[('gb',
                 HistGradientBoostingRegressor(early_stopping=False,
                                               l2_regularization=0.1,
                                               learning_rate=0.03, max_iter=500,
                                               min_samples_leaf=50,
                                               random_state=42))])
1.1985170218861214


In [302]:
if bm_val_rmse_top<best_model_val_rmse:
    best_model = best_model_top
    X_test = X_test_top

In [303]:
y_test_pred = best_model.predict(X_test)
best_model_test_rmse = mean_squared_error(y_test, y_test_pred)**.5
print(best_model)
print(best_model_test_rmse)

Pipeline(steps=[('gb',
                 HistGradientBoostingRegressor(early_stopping=False,
                                               l2_regularization=0.1,
                                               learning_rate=0.03, max_iter=500,
                                               min_samples_leaf=50,
                                               random_state=42))])
1.8096029932022408
